imports

In [2]:
import clip
import json
import requests
import torch
import faiss
from tqdm import tqdm
import numpy as np
from PIL import Image
from io import BytesIO

In [3]:
with open('data/annotations/instances_val2017.json', 'r') as file:
    # Parse the JSON file into a Python object
    data = json.load(file)

# print(data["images"][0]["coco_url"])
    
images = data["images"]

def fetch_image(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img

# print(fetch_image(images[0]["coco_url"]).size)
# print(fetch_image(images[0]["coco_url"]).mode)
# fetch_image(images[0]["coco_url"]).show()

model

In [4]:

model, preprocess = clip.load("ViT-B/32", device="cpu")

In [5]:
test_img = fetch_image(images[0]["coco_url"])

preprocess

In [6]:
def embed_image(pil_image): 
    img_tensor = preprocess(pil_image).unsqueeze(0)
    # print(img_tensor.shape)

    with torch.no_grad(): 
        embedding = model.encode_image(img_tensor)

    embedding_np = embedding.detach().cpu().numpy()

    return embedding_np
    # print(embedding.shape)
    # print(embedding_np.shape)


# embed_image(test_img) 

In [7]:
all_embeddings = []
metadata = []

for i in tqdm(range(0, 500, 32)):
    # images[i:i+32]
    batch_tensors = []
    for img in images[i:i+32]:
        try: 
            img_url = img["coco_url"]
            pil_image = fetch_image(img_url)
            embedding = preprocess(pil_image)
            batch_tensors.append(embedding)
            metadata.append({"id": img["id"], "coco_url": img_url})
        except Exception as e: 
            print(f"Skipped {img_url}: {e}")
            continue

    if len(batch_tensors) == 0:
        continue

    stack = torch.stack(batch_tensors)
    # print(stack.shape)

    with torch.no_grad(): 
        embedding = model.encode_image(stack)
        
    embedding_np = embedding.detach().cpu().numpy()
    # print(embedding_np.shape)  
    all_embeddings.append(embedding_np)

embeddings_matrix = np.vstack(all_embeddings)
embeddings_matrix = embeddings_matrix.astype('float32')
print(embeddings_matrix.shape)  

 38%|███▊      | 6/16 [00:49<01:23,  8.39s/it]

In [ ]:
import gc
del all_embeddings
del stack
del embedding
del batch_tensors
import gc
gc.collect()

NameError: name 'all_embeddings' is not defined

In [ ]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")

In [ ]:
faiss.normalize_L2(embeddings_matrix)
line = embeddings_matrix[0].reshape(1, 512)

d = 512

index = faiss.IndexFlatIP(d)
index.add(embeddings_matrix)   
print(index.ntotal)            

query = embeddings_matrix[0].reshape(1, 512)  # search with just one
D, I = index.search(query, 5)
print(D)  # distances — first score should be ~1.0
print(I)  # indices — expected first result is  0 (itself)        

: 